In [152]:
import numpy as np
import pandas as pd
import time
import os
import json

from data.datasets import selecionar_dataset_e_classe, carregar_dataset
from utils.results_handler import update_method_results
from utils.progress_bar import ProgressBar

from tqdm import tqdm  # Alterado de tqdm.notebook para usar a versão de texto, evitando o erro de IProgress
# Imports do Scikit-learn para o novo experimento
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier  # MLP
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance # <-- Para a nova heurística

RANDOM_STATE = 42


In [153]:

# 02 Constantes e funções de configuração reutilizadas do peab.py

# Configurações específicas de MNIST (opcional, mas bom ter para consistência)
MNIST_CONFIG = {
    'feature_mode': 'raw',
    'digit_pair': (3, 8),
    'top_k_features': None,
    'test_size': 0.3,
    'rejection_cost': 0.24,
    'subsample_size': 0.01   # aplica SOMENTE no X_test (treino permanece completo)
}

# Dicionário principal de configuração dos datasets
DATASET_CONFIG = {
    "mnist":                MNIST_CONFIG,
    "breast_cancer":        {'test_size': 0.3, 'rejection_cost': 0.24},
    "pima_indians_diabetes":{'test_size': 0.3, 'rejection_cost': 0.24},
    "vertebral_column":     {'test_size': 0.3, 'rejection_cost': 0.24},
    "sonar":                {'test_size': 0.3, 'rejection_cost': 0.24},
    "spambase":             {'test_size': 0.3, 'rejection_cost': 0.24},
    "banknote":             {'test_size': 0.3, 'rejection_cost': 0.24},
    "heart_disease":        {'test_size': 0.3, 'rejection_cost': 0.24},
    "wine":                 {'subsample_size': 0.20, 'test_size': 0.3, 'rejection_cost': 0.24},
    "creditcard":           {'subsample_size': 0.03, 'test_size': 0.3, 'rejection_cost': 0.040},
    "covertype":            {'subsample_size': 0.005, 'test_size': 0.3, 'rejection_cost': 0.24},
    "gas_sensor":           {'subsample_size': 0.05, 'test_size': 0.3, 'rejection_cost': 0.045},
    "newsgroups":           {'subsample_size': 0.5, 'test_size': 0.3, 'rejection_cost': 0.24},
    "rcv1":                 {'subsample_size': 0.5, 'test_size': 0.3, 'rejection_cost': 0.24},
}

def configurar_experimento(dataset_name: str):
    """Carrega o dataset e as configurações específicas para ele."""
    if dataset_name == 'mnist':
        from data import datasets as ds_module
        cfg = DATASET_CONFIG.get(dataset_name, {})
        ds_module.set_mnist_options(cfg.get('feature_mode', 'raw'), cfg.get('digit_pair', None))
    
    X, y, nomes_classes = carregar_dataset(dataset_name)
    cfg = DATASET_CONFIG.get(dataset_name, {'test_size': 0.3, 'rejection_cost': 0.24})

    subsample_size = cfg.get('subsample_size', None)  # None = usa todo o X_test

    return X, y, nomes_classes, cfg['rejection_cost'], cfg['test_size'], subsample_size


In [154]:

# --- SELEÇÃO DO DATASET ---
# 1. Liste os datasets disponíveis
available_datasets = list(DATASET_CONFIG.keys())
print("Datasets disponíveis:")
for i, name in enumerate(available_datasets):
    print(f"  {i}: {name}")

# 2. Escolha o dataset pelo número (índice)
"""Datasets disponíveis:
  0: mnist
  1: breast_cancer
  2: pima_indians_diabetes
  3: vertebral_column
  4: sonar
  5: spambase
  6: banknote
  7: heart_disease
  8: wine
  9: creditcard
  10: covertype
  11: gas_sensor
  12: newsgroups
  13: rcv1"""
dataset_index = 2  # escolher o numero relacionado ao dataset desejado
DATASET_NAME = available_datasets[dataset_index]
# --- FIM DA SELEÇÃO ---

print(f"\n--> Dataset selecionado: '{DATASET_NAME}'\n")

X, y, nomes_classes, rejection_cost, test_size, subsample_size = configurar_experimento(DATASET_NAME)

# Treino usa SEMPRE 100% dos dados (sem subsampling) para o modelo ficar bem treinado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
)

# Subsampling aplicado SOMENTE no X_test (geração de explicações)
# O treino permanece completo independente do subsample_size configurado
x_test_original_len = len(X_test)
if subsample_size is not None and subsample_size < 1.0:
    n_subsample = max(1, int(x_test_original_len * subsample_size))
    rng = np.random.default_rng(RANDOM_STATE)
    subsample_idx = rng.choice(x_test_original_len, size=n_subsample, replace=False)
    subsample_idx.sort()
    X_test = X_test.iloc[subsample_idx]
    y_test = y_test.iloc[subsample_idx]
    print(f"[subsample] X_test reduzido de {x_test_original_len} -> {len(X_test)} instâncias ({subsample_size*100:.1f}% do teste)")
    print(f"[treino]    X_train mantido com {X_train.shape[0]} instâncias (completo)\n")
else:
    print(f"[subsample] Nenhum subsampling aplicado — usando todo o X_test ({x_test_original_len} instâncias)\n")

print(f"Dataset '{DATASET_NAME}' carregado.")
print(f"Treino: {X_train.shape[0]} instâncias | Teste (explicações): {X_test.shape[0]} instâncias")


Datasets disponíveis:
  0: mnist
  1: breast_cancer
  2: pima_indians_diabetes
  3: vertebral_column
  4: sonar
  5: spambase
  6: banknote
  7: heart_disease
  8: wine
  9: creditcard
  10: covertype
  11: gas_sensor
  12: newsgroups
  13: rcv1

--> Dataset selecionado: 'pima_indians_diabetes'

[subsample] Nenhum subsampling aplicado — usando todo o X_test (231 instâncias)

Dataset 'pima_indians_diabetes' carregado.
Treino: 537 instâncias | Teste (explicações): 231 instâncias


In [155]:
# 03: Treinar o modelo MLP (novo experimento)

def treinar_modelo_mlp(X_train, y_train, mlp_params):
    """
    Cria e treina um pipeline com MinMaxScaler e MLPClassifier.
    """
    pipeline = Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', MLPClassifier(random_state=RANDOM_STATE, **mlp_params))
    ])
    
    print("Treinando o modelo MLP...")
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    end_time = time.time()
    print(f"Treinamento concluído em {end_time - start_time:.2f} segundos.")
    
    return pipeline

# Parâmetros MLP.
MLP_PARAMS = {
    'hidden_layer_sizes': (100, 50),  # Duas camadas ocultas com 100 e 50 neurônios
    'activation': 'relu',
    'solver': 'adam',
    'max_iter': 500,                 # numero maximo de epocas que o modelo pode treinar
    'alpha': 0.0001,                 # Termo de regularização L2
    'learning_rate_init': 0.001,     # taxa de aprendizado incial
    'early_stopping': True,          # True para parar quando o backpropagation não conseguir mais ajustar
    'n_iter_no_change': 10,          # Nº de iterações sem melhora antes de parar
    'validation_fraction': 0.1       # Fração do treino usada para validação do early stopping
}

# Treinando o modelo para ter um objeto para os próximos passos
print("Treinando o modelo MLP... (para ter o objeto 'modelo_mlp' disponível)")
modelo_mlp = treinar_modelo_mlp(X_train, y_train, MLP_PARAMS)

# A lógica de encontrar os thresholds (t+ e t-) continua a mesma,
# basta usar o `modelo_mlp.decision_function()` no seu conjunto de validação.


Treinando o modelo MLP... (para ter o objeto 'modelo_mlp' disponível)
Treinando o modelo MLP...
Treinamento concluído em 0.11 segundos.


In [156]:
def check_validity_model_agnostic(
    fixed_indices: set,
    instance_vals: np.ndarray,        # MUDANÇA: recebe numpy array direto
    modelo: Pipeline,
    bounds: dict,                      # MUDANÇA: recebe bounds pré-computados
    t_plus: float,
    t_minus: float,
    mode: str
) -> bool:
    """
    Verifica a validade de uma explicação para um modelo caixa-preta (MLP).
    Versão otimizada: opera com numpy arrays e faz UMA chamada batch ao predict_proba.
    """
    min_orig = bounds['min_orig']
    max_orig = bounds['max_orig']
    feature_names = bounds['feature_names']
    n_features = len(instance_vals)

    # Cria cópias numpy (muito mais rápido que pd.Series.copy())
    inst_min_case = instance_vals.copy()
    inst_max_case = instance_vals.copy()

    # Perturba features NÃO fixadas para os extremos
    for i in range(n_features):
        if i not in fixed_indices:
            inst_min_case[i] = min_orig[i]
            inst_max_case[i] = max_orig[i]

    # Cria DataFrame de batch com as 2 instâncias perturbadas (1 chamada ao pipeline)
    df_batch = pd.DataFrame([inst_min_case, inst_max_case], columns=feature_names)
    probas_batch = np.clip(modelo.predict_proba(df_batch), 1e-9, 1 - 1e-9)
    scores_batch = np.log(probas_batch[:, 1] / probas_batch[:, 0])

    score_min_case = scores_batch[0]
    score_max_case = scores_batch[1]

    EPSILON = 1e-5
    if mode == 'positive':
        worst_score = min(score_min_case, score_max_case)
        return worst_score >= t_plus - EPSILON

    elif mode == 'negative':
        best_adversary_score = max(score_min_case, score_max_case)
        return best_adversary_score <= t_minus + EPSILON

    elif mode == 'rejected':
        worst_score = min(score_min_case, score_max_case)
        best_adversary_score = max(score_min_case, score_max_case)
        return (worst_score >= t_minus - EPSILON) and (best_adversary_score <= t_plus + EPSILON)

    return False

In [157]:
def preparar_bounds_mlp(modelo: Pipeline, X_train: pd.DataFrame) -> dict:
    """
    Pré-computa os limites min/max do treino UMA VEZ fora do loop.
    Evita recalcular X_train.min() a cada chamada de check_validity.
    """
    min_orig = X_train.values.min(axis=0)
    max_orig = X_train.values.max(axis=0)
    return {
        'min_orig': min_orig,
        'max_orig': max_orig,
        'feature_names': X_train.columns.tolist()
    }


def calcular_sorting_metric_local(
    instance_vals: np.ndarray,
    modelo: Pipeline,
    feature_names: list,
    delta_frac: float = 0.01
) -> np.ndarray:
    """
    Calcula importância LOCAL de cada feature para UMA instância via gradiente numérico.

    Vantagens sobre Permutation Importance global:
    - SEM data leakage (não usa X_test)
    - Muito mais rápido: ~n_features chamadas vs. n_repeats * n_samples
    - Mais preciso: importância local à instância, ideal para explicações individuais
    """
    n_features = len(instance_vals)

    # Score base da instância original
    df_base = pd.DataFrame([instance_vals], columns=feature_names)
    probas_base = np.clip(modelo.predict_proba(df_base)[0], 1e-9, 1 - 1e-9)
    score_base = np.log(probas_base[1] / probas_base[0])

    # Cria batch com todas as perturbações de uma vez (1 chamada ao pipeline)
    perturbed_batch = np.tile(instance_vals, (n_features, 1))
    for j in range(n_features):
        delta = max(abs(instance_vals[j]) * delta_frac, 1e-6)
        perturbed_batch[j, j] = instance_vals[j] + delta

    df_batch = pd.DataFrame(perturbed_batch, columns=feature_names)
    probas_batch = np.clip(modelo.predict_proba(df_batch), 1e-9, 1 - 1e-9)
    scores_batch = np.log(probas_batch[:, 1] / probas_batch[:, 0])

    importancias = np.abs(scores_batch - score_base)
    return importancias


# Pré-computa bounds UMA VEZ (fora do loop principal)
bounds_mlp = preparar_bounds_mlp(modelo_mlp, X_train)
feature_names_list = bounds_mlp['feature_names']

print("Bounds pré-computados com sucesso.")
print(f"  Exemplo min[0]={bounds_mlp['min_orig'][0]:.4f} | max[0]={bounds_mlp['max_orig'][0]:.4f}")
print(f"  Features: {len(feature_names_list)}")

Bounds pré-computados com sucesso.
  Exemplo min[0]=0.0000 | max[0]=17.0000
  Features: 8


In [158]:
from sklearn.utils import resample
from sklearn.metrics import classification_report

def detectar_e_corrigir_colapso(modelo, X_train, y_train, X_test, y_test, mlp_params, random_state=42):
    """
    Detecta colapso de classe no MLP e retreina com dados balanceados se necessário.
    Colapso = modelo prevê apenas 1 classe em >95% do teste.
    """
    preds = modelo.predict(X_test)
    classes, counts = np.unique(preds, return_counts=True)
    taxa_dominante = counts.max() / counts.sum()

    print(f"[colapso check] Classes preditas: {dict(zip(classes, counts))}")
    print(f"[colapso check] Taxa da classe dominante: {taxa_dominante:.1%}")

    if taxa_dominante < 0.95:
        print("[colapso check] ✅ Sem colapso detectado — modelo OK.")
        return modelo, X_train, y_train  # retorna sem mudança

    print("[colapso check] ⚠️  Colapso detectado! Aplicando balanceamento por oversampling...")

    # Balanceia X_train por oversampling da classe minoritária
    classes_orig, counts_orig = np.unique(y_train, return_counts=True)
    n_max = counts_orig.max()
    X_parts, y_parts = [], []

    X_df = pd.DataFrame(X_train) if not isinstance(X_train, pd.DataFrame) else X_train.copy()
    y_s  = pd.Series(y_train.values if hasattr(y_train, 'values') else y_train)

    for cls in classes_orig:
        mask = (y_s == cls)
        Xc, yc = X_df[mask.values], y_s[mask.values]
        if len(Xc) < n_max:
            Xc, yc = resample(Xc, yc, replace=True, n_samples=n_max, random_state=random_state)
        X_parts.append(Xc)
        y_parts.append(yc)

    X_bal = pd.concat(X_parts).reset_index(drop=True)
    y_bal = pd.concat(y_parts).reset_index(drop=True)

    print(f"[balanceamento] Antes: {dict(zip(classes_orig, counts_orig))}")
    classes_bal, counts_bal = np.unique(y_bal, return_counts=True)
    print(f"[balanceamento] Depois: {dict(zip(classes_bal, counts_bal))}")

    # Parâmetros ajustados para retreino balanceado
    mlp_params_bal = {**mlp_params,
        'max_iter': 1000,
        'learning_rate_init': 0.0005,
        'n_iter_no_change': 20,
        'validation_fraction': 0.15,
    }

    print("[balanceamento] Retreinando MLP com dados balanceados...")
    modelo_bal = treinar_modelo_mlp(X_bal, y_bal, mlp_params_bal)

    # Verifica se corrigiu
    preds_bal = modelo_bal.predict(X_test)
    print("\n[após balanceamento] Classification report:")
    print(classification_report(y_test, preds_bal))

    return modelo_bal, X_bal, y_bal


# Detecta colapso e corrige automaticamente se necessário
# X_train_orig preservado para bounds (distribuição real)
X_train_orig = X_train.copy()

modelo_mlp, X_train_para_threshold, y_train_para_threshold = detectar_e_corrigir_colapso(
    modelo_mlp, X_train, y_train, X_test, y_test, MLP_PARAMS
)

# bounds sempre calculados sobre X_train ORIGINAL (não o balanceado)
bounds_mlp = preparar_bounds_mlp(modelo_mlp, X_train_orig)
feature_names_list = bounds_mlp['feature_names']
print(f"\nBounds recalculados sobre X_train original ({X_train_orig.shape[0]} inst.)")

[colapso check] Classes preditas: {0: 231}
[colapso check] Taxa da classe dominante: 100.0%
[colapso check] ⚠️  Colapso detectado! Aplicando balanceamento por oversampling...
[balanceamento] Antes: {0: 350, 1: 187}
[balanceamento] Depois: {0: 350, 1: 350}
[balanceamento] Retreinando MLP com dados balanceados...
Treinando o modelo MLP...
Treinamento concluído em 0.20 segundos.

[após balanceamento] Classification report:
              precision    recall  f1-score   support

           0       0.84      0.61      0.71       150
           1       0.52      0.78      0.62        81

    accuracy                           0.67       231
   macro avg       0.68      0.70      0.67       231
weighted avg       0.73      0.67      0.68       231


Bounds recalculados sobre X_train original (537 inst.)


In [159]:
def encontrar_thresholds_otimos(X_train, y_train, rejection_cost, mlp_params, val_size=0.2,
                                 modelo_fixo=None):  # <-- NOVO PARÂMETRO
    """
    Encontra os thresholds t+ e t- ótimos em um conjunto de validação.
    Se modelo_fixo for passado, usa ele direto (sem retreinar).
    """
    print("Encontrando thresholds ótimos (t+ e t-)...")
    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train, y_train, test_size=val_size, random_state=RANDOM_STATE, stratify=y_train
    )

    # FIX: usa o modelo já treinado se disponível (evita retreino com scores espalhados)
    if modelo_fixo is not None:
        modelo_temp = modelo_fixo
        print("  [threshold] Usando modelo já treinado (sem retreino).")
    else:
        modelo_temp = treinar_modelo_mlp(X_train_sub, y_train_sub, mlp_params)

    # Calcula scores no conjunto de validação
    probas = modelo_temp.predict_proba(X_val)
    epsilon = 1e-9
    probas = np.clip(probas, epsilon, 1 - epsilon)
    decision_scores = np.log(probas[:, 1] / probas[:, 0])

    print(f"  [threshold] Scores val: min={decision_scores.min():.4f} | max={decision_scores.max():.4f}")

    scores_neg = decision_scores[decision_scores < 0]
    scores_pos = decision_scores[decision_scores > 0]

    t_minus_grid = np.linspace(scores_neg.min(), -0.0001, 50) if len(scores_neg) > 0 else np.array([-0.1])
    t_plus_grid  = np.linspace(0.0001, scores_pos.max(), 50)  if len(scores_pos) > 0 else np.array([0.1])

    best_risk = float('inf')
    best_t_plus, best_t_minus = 0.1, -0.1

    for tm in t_minus_grid:
        for tp in t_plus_grid:
            if not (tm < 0 < tp): continue

            accepted_mask = (decision_scores >= tp) | (decision_scores <= tm)
            preds = np.full(y_val.shape, -1)
            preds[decision_scores >= tp] = 1
            preds[decision_scores <= tm] = 0

            error = np.mean(preds[accepted_mask] != y_val.values[accepted_mask]) if np.any(accepted_mask) else 0.0
            rejection_rate = 1.0 - np.mean(accepted_mask)
            risk = error + rejection_cost * rejection_rate

            if risk < best_risk:
                best_risk, best_t_plus, best_t_minus = risk, tp, tm

    print(f"Thresholds encontrados: t+={best_t_plus:.4f}, t-={best_t_minus:.4f} (Risco: {best_risk:.4f})")

    # Alerta se zona de rejeição for muito grande (>80% das instâncias rejeitadas)
    rej_rate_check = np.mean((decision_scores > best_t_minus) & (decision_scores < best_t_plus))
    if rej_rate_check > 0.8:
        print(f"  ⚠️  ALERTA: zona de rejeição muito grande ({rej_rate_check:.1%} rejeitadas na validação)!")
        print(f"  ⚠️  Considere aumentar rejection_cost no DATASET_CONFIG para este dataset.")

    return float(best_t_plus), float(best_t_minus)


# Encontrar os thresholds passando o modelo já treinado
t_plus, t_minus = encontrar_thresholds_otimos(
    X_train_para_threshold, y_train_para_threshold,
    rejection_cost, MLP_PARAMS,
    modelo_fixo=modelo_mlp        # <-- PASSA O MODELO JÁ TREINADO
)

Encontrando thresholds ótimos (t+ e t-)...
  [threshold] Usando modelo já treinado (sem retreino).
  [threshold] Scores val: min=-0.2225 | max=0.3654
Thresholds encontrados: t+=0.1492, t-=-0.1090 (Risco: 0.2435)


In [160]:
def fase_1_reforco_agnostic(
    modelo: Pipeline,
    instance_vals: np.ndarray,        # MUDANÇA: numpy array
    bounds: dict,                      # MUDANÇA: bounds pré-computados
    t_plus: float,
    t_minus: float,
    mode: str,
    sorting_metric: np.ndarray
) -> set:
    expl_indices = set()
    indices_ordenados = np.argsort(-sorting_metric)

    for idx in indices_ordenados:
        if check_validity_model_agnostic(expl_indices, instance_vals, modelo, bounds, t_plus, t_minus, mode):
            break
        if idx not in expl_indices:
            expl_indices.add(int(idx))

    # Segurança: se ainda inválido com todas as features, retorna tudo
    if not check_validity_model_agnostic(expl_indices, instance_vals, modelo, bounds, t_plus, t_minus, mode):
        return set(range(len(instance_vals)))

    return expl_indices


def fase_2_minimizacao_agnostic(
    modelo: Pipeline,
    instance_vals: np.ndarray,        # MUDANÇA: numpy array
    expl_indices_inicial: set,
    bounds: dict,                      # MUDANÇA: bounds pré-computados
    t_plus: float,
    t_minus: float,
    mode: str,
    sorting_metric: np.ndarray
) -> set:
    expl_indices = expl_indices_inicial.copy()
    features_presentes = sorted(list(expl_indices), key=lambda i: sorting_metric[i])

    for idx in features_presentes:
        if len(expl_indices) <= 1:
            break
        expl_indices.discard(idx)
        if not check_validity_model_agnostic(expl_indices, instance_vals, modelo, bounds, t_plus, t_minus, mode):
            expl_indices.add(idx)

    return expl_indices


def gerar_explicacao_instancia_mlp(
    instance_vals: np.ndarray,
    modelo: Pipeline,
    bounds: dict,
    t_plus: float,
    t_minus: float,
    feature_names: list
) -> list:
    """
    Gera explicação abdutiva para UMA instância do MLP.
    Sorting metric calculada localmente por gradiente numérico (sem data leakage).
    """
    df_inst = pd.DataFrame([instance_vals], columns=feature_names)
    probas = np.clip(modelo.predict_proba(df_inst)[0], 1e-9, 1 - 1e-9)
    score_raw = np.log(probas[1] / probas[0])

    if score_raw >= t_plus:
        mode = 'positive'
    elif score_raw <= t_minus:
        mode = 'negative'
    else:
        mode = 'rejected'

    sorting_metric = calcular_sorting_metric_local(instance_vals, modelo, feature_names)

    indices_robustos = fase_1_reforco_agnostic(
        modelo, instance_vals, bounds, t_plus, t_minus, mode, sorting_metric
    )

    indices_minimos = fase_2_minimizacao_agnostic(
        modelo, instance_vals, indices_robustos, bounds, t_plus, t_minus, mode, sorting_metric
    )

    # FIX: explicação vazia = modelo muito confiante, mas semanticamente inútil
    # Garante pelo menos 1 feature (a mais importante localmente)
    if len(indices_minimos) == 0:
        indices_minimos = {int(np.argmax(sorting_metric))}

    return [feature_names[i] for i in sorted(list(indices_minimos))]

In [161]:
# --- CÉLULA DE EXECUÇÃO E COLETA DE DADOS ---

print(f"Iniciando a geração de explicações para {X_test.shape[0]} instâncias de teste...")
start_time_total = time.time()

per_instance_results = []

# Pré-cálculo das predições e scores para todo o conjunto de teste
probas = np.clip(modelo_mlp.predict_proba(X_test), 1e-9, 1 - 1e-9)
scores = np.log(probas[:, 1] / probas[:, 0])

preds_sem_rejeicao = modelo_mlp.predict(X_test)

preds_com_rejeicao = np.full(len(X_test), 2)
preds_com_rejeicao[scores >= t_plus] = 1
preds_com_rejeicao[scores <= t_minus] = 0

# MUDANÇA: extrai X_test como numpy UMA VEZ fora do loop
X_test_vals = X_test.values
feature_names_list = X_train.columns.tolist()

# Loop principal com barra de progresso
for i in tqdm(range(len(X_test)), desc="Gerando Explicações MLP"):
    inst_vals = X_test_vals[i]          # numpy array (sem overhead do .iloc)
    start_inst_time = time.time()

    # MUDANÇA: passa numpy array + bounds pré-computados
    explicacao = gerar_explicacao_instancia_mlp(
        inst_vals, modelo_mlp, bounds_mlp, t_plus, t_minus, feature_names_list
    )

    end_inst_time = time.time()

    per_instance_results.append({
        'id': str(X_test.index[i]),
        'y_true': int(y_test.iloc[i]),
        'y_pred_sem_rejeicao': int(preds_sem_rejeicao[i]),
        'y_pred_com_rejeicao': int(preds_com_rejeicao[i]),
        'decision_score': float(scores[i]),
        'explicacao': explicacao,
        'tamanho_explicacao': len(explicacao),
        'tempo_execucao': end_inst_time - start_inst_time
    })

end_time_total = time.time()
total_execution_time = end_time_total - start_time_total
print(f"\nGeração de explicações concluída em {total_execution_time:.2f} segundos.")

# --- AGREGAÇÃO E SALVAMENTO EM JSON ---

mask_rej = (preds_com_rejeicao == 2)
acc_sem_rejeicao = np.mean(preds_sem_rejeicao == y_test) * 100
acc_com_rejeicao = np.mean(preds_com_rejeicao[~mask_rej] == y_test.values[~mask_rej]) * 100 if np.any(~mask_rej) else 100.0

tamanhos_pos = [r['tamanho_explicacao'] for r in per_instance_results if r['y_pred_com_rejeicao'] == 1]
tamanhos_neg = [r['tamanho_explicacao'] for r in per_instance_results if r['y_pred_com_rejeicao'] == 0]
tamanhos_rej = [r['tamanho_explicacao'] for r in per_instance_results if r['y_pred_com_rejeicao'] == 2]

def calc_stats(data_list):
    if not data_list: return {'count': 0, 'mean': 0, 'std': 0, 'min': 0, 'max': 0}
    return {
        'count': len(data_list),
        'mean': float(np.mean(data_list)),
        'std': float(np.std(data_list)),
        'min': int(np.min(data_list)),
        'max': int(np.max(data_list))
    }

final_results = {
    'config': {
        'dataset_name': DATASET_NAME,
        'num_instances_total': X.shape[0],
        'num_features': X.shape[1],
        'test_size': test_size,
        'rejection_cost': rejection_cost,
        'random_state': RANDOM_STATE
    },
    'model_params': {
        'model_type': 'MLPClassifier',
        'hidden_layer_sizes': MLP_PARAMS['hidden_layer_sizes'],
        'activation': MLP_PARAMS['activation'],
        'solver': MLP_PARAMS['solver'],
        'max_iter': MLP_PARAMS['max_iter'],
        'n_iter_': modelo_mlp.named_steps['model'].n_iter_,
        'n_layers_': modelo_mlp.named_steps['model'].n_layers_,
    },
    'thresholds': {
        't_plus': t_plus,
        't_minus': t_minus,
        'rejection_zone_width': t_plus - t_minus
    },
    'performance': {
        'accuracy_without_rejection': acc_sem_rejeicao,
        'accuracy_with_rejection': acc_com_rejeicao,
        'rejection_rate': np.mean(mask_rej) * 100,
        'num_test_instances': len(X_test),
        'num_positive': len(tamanhos_pos),
        'num_negative': len(tamanhos_neg),
        'num_rejected': len(tamanhos_rej),
    },
    'explanation_stats': {
        'positive': calc_stats(tamanhos_pos),
        'negative': calc_stats(tamanhos_neg),
        'rejected': calc_stats(tamanhos_rej)
    },
    'computation_time': {
        'total_seconds': total_execution_time,
        'mean_per_instance_seconds': total_execution_time / len(X_test) if len(X_test) > 0 else 0
    },
    'per_instance': per_instance_results
}

output_dir_json = 'json/MLP'
os.makedirs(output_dir_json, exist_ok=True)
json_filepath = os.path.join(output_dir_json, f'{DATASET_NAME}.json')

with open(json_filepath, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=4)

print(f"Resultados salvos em: {json_filepath}")

Iniciando a geração de explicações para 231 instâncias de teste...


Gerando Explicações MLP: 100%|██████████| 231/231 [00:02<00:00, 92.04it/s]


Geração de explicações concluída em 2.51 segundos.
Resultados salvos em: json/MLP\pima_indians_diabetes.json


In [162]:
# --- CÉLULA DE GERAÇÃO DE RELATÓRIO A PARTIR DO JSON ---

def gerar_relatorio_texto_mlp(json_filepath: str):
    """
    Gera um relatório de texto formatado a partir de um arquivo JSON de resultados do MLP.
    """
    # Carregar dados do JSON
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Criar diretório de saída
    output_dir_report = 'results/report_MLP'
    os.makedirs(output_dir_report, exist_ok=True)
    report_filepath = os.path.join(output_dir_report, f"report_{data['config']['dataset_name']}.txt")

    # Extrair seções do dicionário para facilitar o acesso
    cfg = data['config']
    model = data['model_params']
    thresh = data['thresholds']
    perf = data['performance']
    exp_stats = data['explanation_stats']
    comp_time = data['computation_time']

    with open(report_filepath, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write(f"          RELATÓRIO DE ANÁLISE - MÉTODO MLP COM REJEIÇÃO\n")
        f.write("="*80 + "\n\n")

        # 1. Resumo do Experimento
        f.write("1. RESUMO DO EXPERIMENTO\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Nome do Dataset: {cfg['dataset_name']}\n")
        f.write(f"  - Total de Instâncias: {cfg['num_instances_total']}\n")
        f.write(f"  - Total de Features: {cfg['num_features']}\n")
        f.write(f"  - Divisão Treino/Teste: {1-cfg['test_size']:.0%}/{cfg['test_size']:.0%}\n")
        f.write(f"  - Instâncias de Treino: {cfg['num_instances_total'] - perf['num_test_instances']}\n")
        f.write(f"  - Instâncias de Teste: {perf['num_test_instances']}\n\n")

        # 2. Configuração do Modelo MLP
        f.write("2. CONFIGURAÇÃO DO MODELO (MLP)\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Neurônios nas Camadas Ocultas: {model['hidden_layer_sizes']}\n")
        f.write(f"  - Função de Ativação: {model['activation']}\n")
        f.write(f"  - Solver: {model['solver']}\n")
        f.write(f"  - Épocas de Treinamento (Backpropagation): {model['n_iter_']}\n\n")

        # 3. Zona de Rejeição
        f.write("3. ZONA DE REJEIÇÃO\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Custo de Rejeição (usado para otimização): {cfg['rejection_cost']}\n")
        f.write(f"  - Limiar Superior (t+): {thresh['t_plus']:.4f}\n")
        f.write(f"  - Limiar Inferior (t-): {thresh['t_minus']:.4f}\n")
        f.write(f"  - Tamanho da Zona de Rejeição: {thresh['rejection_zone_width']:.4f}\n\n")

        # 4. Desempenho
        f.write("4. DESEMPENHO DO CLASSIFICADOR\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Acurácia ANTES da Rejeição: {perf['accuracy_without_rejection']:.2f}%\n")
        f.write(f"  - Acurácia DEPOIS da Rejeição (nas aceitas): {perf['accuracy_with_rejection']:.2f}%\n")
        f.write(f"  - Taxa de Rejeição: {perf['rejection_rate']:.2f}%\n")
        f.write(f"  - Instâncias Positivas: {perf['num_positive']}\n")
        f.write(f"  - Instâncias Negativas: {perf['num_negative']}\n")
        f.write(f"  - Instâncias Rejeitadas: {perf['num_rejected']}\n\n")

        # 5. Estatísticas das Explicações
        f.write("5. ESTATÍSTICAS DAS EXPLICAÇÕES\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Para Instâncias POSITIVAS ({exp_stats['positive']['count']}):\n")
        f.write(f"    - Tamanho Médio da Explicação: {exp_stats['positive']['mean']:.2f} features\n")
        f.write(f"  - Para Instâncias NEGATIVAS ({exp_stats['negative']['count']}):\n")
        f.write(f"    - Tamanho Médio da Explicação: {exp_stats['negative']['mean']:.2f} features\n")
        f.write(f"  - Para Instâncias REJEITADAS ({exp_stats['rejected']['count']}):\n")
        f.write(f"    - Tamanho Médio da Explicação: {exp_stats['rejected']['mean']:.2f} features\n\n")

        # 6. Tempo de Execução
        f.write("6. TEMPO DE EXECUÇÃO (Geração das Explicações)\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Tempo Total: {comp_time['total_seconds']:.2f} segundos\n")
        f.write(f"  - Tempo Médio por Instância: {comp_time['mean_per_instance_seconds']:.4f} segundos\n\n")
        
    print(f"Relatório de texto gerado em: {report_filepath}")

# Chamar a função para gerar o relatório usando o arquivo JSON salvo anteriormente
gerar_relatorio_texto_mlp(json_filepath)

Relatório de texto gerado em: results/report_MLP\report_pima_indians_diabetes.txt


In [163]:
# Célula de diagnóstico — rode no notebook
print("=== DIAGNÓSTICO THRESHOLDS PIMA ===")
probas_diag = modelo_mlp.predict_proba(X_test)
probas_diag = np.clip(probas_diag, 1e-9, 1-1e-9)
scores_diag = np.log(probas_diag[:,1] / probas_diag[:,0])

print(f"Scores no X_test:")
print(f"  min={scores_diag.min():.4f} | max={scores_diag.max():.4f} | mean={scores_diag.mean():.4f}")
print(f"  std={scores_diag.std():.4f}")
print(f"\nt+ atual = {t_plus:.4f} | t- atual = {t_minus:.4f}")
print(f"Zona de rejeição: {t_plus - t_minus:.4f}")
print(f"\nInstâncias acima de t+:  {(scores_diag >= t_plus).sum()}")
print(f"Instâncias abaixo de t-: {(scores_diag <= t_minus).sum()}")
print(f"Instâncias REJEITADAS:   {((scores_diag > t_minus) & (scores_diag < t_plus)).sum()}")

# Verifica distribuição de predições (sem rejeição)
preds_diag = modelo_mlp.predict(X_test)
from sklearn.metrics import classification_report
print(f"\nClassification report (sem rejeição):")
print(classification_report(y_test, preds_diag))

=== DIAGNÓSTICO THRESHOLDS PIMA ===
Scores no X_test:
  min=-0.2401 | max=0.5064 | mean=0.0144
  std=0.1333

t+ atual = 0.1492 | t- atual = -0.1090
Zona de rejeição: 0.2582

Instâncias acima de t+:  40
Instâncias abaixo de t-: 47
Instâncias REJEITADAS:   144

Classification report (sem rejeição):
              precision    recall  f1-score   support

           0       0.84      0.61      0.71       150
           1       0.52      0.78      0.62        81

    accuracy                           0.67       231
   macro avg       0.68      0.70      0.67       231
weighted avg       0.73      0.67      0.68       231



In [164]:
# Diagnóstico MNIST - rode após treinar o modelo
print("=== DIAGNÓSTICO MNIST - THRESHOLDS ===")

# 1. Distribuição das classes no treino e teste
print(f"\n[Distribuição do Dataset]")
print(f"Treino: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Teste:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

# 2. Scores do modelo no X_test
probas_mnist = np.clip(modelo_mlp.predict_proba(X_test), 1e-9, 1-1e-9)
scores_mnist = np.log(probas_mnist[:,1] / probas_mnist[:,0])

print(f"\n[Scores no X_test]")
print(f"  min={scores_mnist.min():.4f} | max={scores_mnist.max():.4f}")
print(f"  mean={scores_mnist.mean():.4f} | std={scores_mnist.std():.4f}")
print(f"  mediana={np.median(scores_mnist):.4f}")

# 3. Thresholds encontrados
print(f"\n[Thresholds]")
print(f"  t+ = {t_plus:.6f}")
print(f"  t- = {t_minus:.6f}")
print(f"  Zona de rejeição: {t_plus - t_minus:.6f}")

# 4. Distribuição de predições
print(f"\n[Predições]")
print(f"  Acima de t+ (positivos):  {(scores_mnist >= t_plus).sum()}")
print(f"  Abaixo de t- (negativos): {(scores_mnist <= t_minus).sum()}")
print(f"  Na zona de rejeição:      {((scores_mnist > t_minus) & (scores_mnist < t_plus)).sum()}")

# 5. Acurácia sem rejeição
from sklearn.metrics import accuracy_score, classification_report
preds_mnist = modelo_mlp.predict(X_test)
print(f"\n[Acurácia sem rejeição]: {accuracy_score(y_test, preds_mnist)*100:.2f}%")
print(f"\n[Classification Report]:")
print(classification_report(y_test, preds_mnist))

# 6. Verifica se houve subsampling
if hasattr(X_test, 'shape'):
    print(f"\n[Tamanho do X_test]: {X_test.shape[0]} instâncias")
    if 'subsample_size' in locals() and subsample_size is not None:
        print(f"  Subsample aplicado: {subsample_size*100:.1f}%")

=== DIAGNÓSTICO MNIST - THRESHOLDS ===

[Distribuição do Dataset]
Treino: {0: 350, 1: 187}
Teste:  {0: 150, 1: 81}

[Scores no X_test]
  min=-0.2401 | max=0.5064
  mean=0.0144 | std=0.1333
  mediana=0.0063

[Thresholds]
  t+ = 0.149203
  t- = -0.109007
  Zona de rejeição: 0.258209

[Predições]
  Acima de t+ (positivos):  40
  Abaixo de t- (negativos): 47
  Na zona de rejeição:      144

[Acurácia sem rejeição]: 67.10%

[Classification Report]:
              precision    recall  f1-score   support

           0       0.84      0.61      0.71       150
           1       0.52      0.78      0.62        81

    accuracy                           0.67       231
   macro avg       0.68      0.70      0.67       231
weighted avg       0.73      0.67      0.68       231


[Tamanho do X_test]: 231 instâncias
